# 17 — Final Model Training and Artifact Preparation

This notebook finalizes the selected fraud detection model workflow and prepares the project for inference and FastAPI integration. The focus is production readiness: clear paths, final artifact locations, and a clean setup for training and saving the validated model in later phases.


## Purpose

This notebook exists to prepare the final training and artifact-saving workflow for the fraud detection system.

It will eventually be responsible for:

- Training the final selected fraud detection model
- Reusing the final decision policy from previous notebooks
- Saving model artifacts needed for inference
- Saving feature columns, metrics, metadata, and decision policy
- Preparing the project for the next inference/API stage

This notebook should not repeat full EDA, full model comparison, or threshold tuning. Those tasks were completed earlier in the project. Notebook 17 is focused on turning the validated modeling decisions into reusable production-style artifacts.


## Previous Notebook Context

The project workflow leading into this notebook is:

- `13_model_training.ipynb` trained candidate models.
- `14_model_evaluation.ipynb` evaluated model performance and confirmed the strongest model choice.
- `15_threshold_tuning.ipynb` selected suitable fraud probability thresholds using validation/OOF predictions and holdout confirmation.
- `16_decision_logic.ipynb` converted model fraud probabilities into `APPROVE`, `REVIEW`, and `BLOCK` decisions.
- `17_final_model_training.ipynb` now prepares the final validated model training and artifact-saving workflow.

The difference between notebook 13 and notebook 17 is important. Notebook 13 is for experimentation and candidate model training. Notebook 17 is for final validated model training and artifact preparation for inference/API use. In other words, notebook 13 helps decide what works; notebook 17 prepares the final version for reuse by downstream code.


## Imports

This section imports the libraries needed for the notebook setup and the later final-training workflow. The imports are intentionally placed up front so the notebook can train the final model, compute metrics, and save artifacts without changing the environment configuration.


In [11]:
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Path Configuration

This section defines all important input and output paths with `pathlib.Path`. The project root logic works whether the notebook is run from the repository root or from inside the `notebooks/` folder.

This setup step only configures paths. It does not load the dataset, train a model, evaluate a model, or save final model artifacts.


In [12]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables" / "17_final_model_training"
FIGURES_DIR = REPORTS_DIR / "figures" / "17_final_model_training"

# Existing processed dataset selected for final model training setup.
# The repository currently contains final_features.csv rather than creditcard_model_ready.csv.
FINAL_DATASET_PATH = DATA_PROCESSED_DIR / "final_features.csv"
DECISION_POLICY_PATH = ARTIFACTS_DIR / "decision_policy.json"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"


## Output Folder Creation

This section creates the folders that later phases will use for artifacts, tables, and figures. Creating them now makes the notebook environment easy to verify before any final model training begins.


In [13]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

configured_paths = {
    "Project root": PROJECT_ROOT,
    "Processed data directory": DATA_PROCESSED_DIR,
    "Artifacts directory": ARTIFACTS_DIR,
    "Tables output directory": TABLES_DIR,
    "Figures output directory": FIGURES_DIR,
    "Final dataset path": FINAL_DATASET_PATH,
    "Decision policy path": DECISION_POLICY_PATH,
    "Final model artifact path": FINAL_MODEL_PATH,
}

print("Configured notebook paths:")
for label, path in configured_paths.items():
    print(f"- {label}: {path}")


Configured notebook paths:
- Project root: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection
- Processed data directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed
- Artifacts directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts
- Tables output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training
- Figures output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/17_final_model_training
- Final dataset path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/final_features.csv
- Decision policy path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/decision_policy.json
- Final model artifact path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib


## Environment Check

Before moving into dataset loading and final model training, confirm the setup questions below:

- Do I know the input dataset path?
- Do I know where final model artifacts will be saved?
- Do I know where reports and tables will be saved?
- Do I understand the difference between notebook 13 and notebook 17?
- Did the required output folders get created successfully?

At this point the notebook environment is configured, but no data has been loaded, no model has been trained, no evaluation has been run, and no final model artifacts have been saved yet.


## Final Model-Ready Dataset Intake

This section loads the final processed dataset that was prepared in earlier notebooks. The purpose is to confirm that the dataset is available and to separate input features from the target column before final model training.


### Load Dataset

The final model-ready dataset should come from the processed data directory configured earlier in this notebook. This section checks that the path exists before loading anything. If the file is missing, the notebook shows the available processed CSV files and stops with a clear error so the dataset path can be corrected safely.


In [14]:
if not FINAL_DATASET_PATH.exists():
    available_files = list(DATA_PROCESSED_DIR.glob("*.csv"))
    print("Available processed CSV files:")
    for file in available_files:
        print("-", file.name)
    raise FileNotFoundError(f"Final dataset not found: {FINAL_DATASET_PATH}")

df = pd.read_csv(FINAL_DATASET_PATH)


### Dataset Shape

This section confirms the size of the final training dataset and shows the first few rows. It helps verify that the notebook is pointing to the expected processed file before any final training logic is added.


In [15]:
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())


Dataset shape: 283726 rows x 14 columns


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount,Class
0,0.192241,-0.311169,-0.097830,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.551600,0.239599,0.025791,5.014760,0
1,-0.153151,-0.143772,-0.053260,1.065235,-0.114805,-0.166974,0.448154,0.463917,0.166480,1.612727,-0.078803,-0.183361,1.305626,0
2,-0.010966,-0.165946,-3.207904,0.066084,1.109969,0.207643,0.379780,-2.890083,1.773209,0.624501,0.791461,-0.121359,5.939276,0
3,-0.051316,-0.287924,0.724897,0.178228,-0.684093,-0.054952,-0.863291,-1.059647,1.792993,-0.226487,0.237609,1.965775,4.824306,0
4,-0.602601,-1.119670,0.107008,0.538196,-0.237033,0.753074,0.403034,-0.451449,1.548718,-0.822843,0.592941,-0.038195,4.262539,0


### Identify Target Column

The fraud target is expected to be stored in the `Class` column.

- `Class = 1` means a fraud transaction.
- `Class = 0` means a normal or non-fraud transaction.

This check keeps the notebook explicit about the prediction target and avoids silent mistakes if the processed dataset schema changes.


In [16]:
TARGET_COL = "Class"

if TARGET_COL not in df.columns:
    print("Available columns:")
    print(df.columns.tolist())
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")


### Separate Features and Target

The final model will eventually learn from the feature matrix `X` to predict the target vector `y`. This phase only performs the separation and prints a compact summary. No train/test split, training, or evaluation happens yet.


In [17]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of feature columns: {X.shape[1]}")
print("First 10 feature columns:")
print(X.columns[:10].tolist())
print("Target distribution:")
print(y.value_counts())


Feature matrix shape: (283726, 13)
Target shape: (283726,)
Number of feature columns: 13
First 10 feature columns:
['V14_V12_interaction', 'V14', 'V17_V16_interaction', 'V12', 'V17', 'V10', 'V4', 'V16', 'V3', 'V11']
Target distribution:
Class
0    283253
1       473
Name: count, dtype: int64


## Dataset Understanding Summary

- `X` contains the input features used by the model.
- `y` contains the correct fraud and non-fraud labels.
- The model will eventually learn patterns from `X` to predict `y`.
- `Class = 1` means fraud.
- `Class = 0` means a normal transaction.
- This step does not validate the full dataset yet; detailed validation can be handled in the next notebook section you choose to add.


## Final Training Dataset Validation

Before final model training, the dataset must be validated to confirm that it is complete, correctly labeled, and structurally consistent. This is important because the final saved model will later be used by the inference pipeline and FastAPI service.


### Missing Value Check

Missing values can break model training or cause the model to learn from incomplete records. For final model training, the dataset should ideally have no missing values.


In [18]:
total_missing_values = int(df.isna().sum().sum())
missing_values_by_column = df.isna().sum()
columns_with_missing_values = missing_values_by_column[missing_values_by_column > 0].sort_values(ascending=False)

print(f"Total missing values: {total_missing_values}")
if columns_with_missing_values.empty:
    print("No missing values found in the final training dataset.")
else:
    display(columns_with_missing_values.rename("missing_value_count").to_frame())


Total missing values: 0
No missing values found in the final training dataset.


### Duplicate Row Check

Duplicate rows can make the model over-learn repeated examples. In a final training dataset, duplicates should be understood before training.


In [19]:
duplicate_rows = int(df.duplicated().sum())
duplicate_percentage = (duplicate_rows / len(df) * 100) if len(df) else 0.0

print(f"Duplicate rows: {duplicate_rows}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")
print("This section reports duplicates only. It does not modify the final training dataset.")


Duplicate rows: 8063
Duplicate percentage: 2.84%
This section reports duplicates only. It does not modify the final training dataset.


### Target Label Validation

The target column must contain only valid binary fraud labels. The expected values are `0` for a normal or non-fraud transaction and `1` for a fraud transaction.


In [20]:
valid_target_values = {0, 1}
actual_target_values = {int(value) for value in df[TARGET_COL].dropna().unique()}
unexpected_values = actual_target_values - valid_target_values

print(f"Unique target values: {sorted(actual_target_values)}")
if unexpected_values:
    raise ValueError(f"Unexpected target values found: {unexpected_values}")

print("Target labels are valid. Only 0 and 1 are present.")


Unique target values: [0, 1]
Target labels are valid. Only 0 and 1 are present.


### Target Distribution and Class Imbalance Summary

Credit card fraud detection is highly imbalanced because fraud transactions are rare compared to normal transactions. This is why accuracy alone is not enough for evaluating the model.


In [21]:
target_counts = y.value_counts().sort_index()
normal_count = int(target_counts.get(0, 0))
fraud_count = int(target_counts.get(1, 0))
total_records = len(y)
normal_percentage = (normal_count / total_records * 100) if total_records else 0.0
fraud_percentage = (fraud_count / total_records * 100) if total_records else 0.0
imbalance_ratio = (normal_count / fraud_count) if fraud_count else float("inf")

target_distribution_summary = pd.DataFrame(
    [
        {
            "class_label": 0,
            "class_meaning": "Normal Transaction",
            "count": normal_count,
            "percentage": normal_percentage,
        },
        {
            "class_label": 1,
            "class_meaning": "Fraud Transaction",
            "count": fraud_count,
            "percentage": fraud_percentage,
        },
    ]
)

display(target_distribution_summary)
print(f"Fraud cases: {fraud_count}")
print(f"Normal cases: {normal_count}")
print(f"Fraud percentage: {fraud_percentage:.4f}%")
print(f"Class imbalance ratio: {imbalance_ratio:.2f} : 1")


,class_label,class_meaning,count,percentage
0,0,Normal Transaction,283253,99.83329
1,1,Fraud Transaction,473,0.16671


Fraud cases: 473
Normal cases: 283253
Fraud percentage: 0.1667%
Class imbalance ratio: 598.84 : 1


### Feature Column Consistency Check

The final training feature matrix should contain only model input columns. The target column must already be removed before any model training starts.


In [22]:
if TARGET_COL in X.columns:
    raise ValueError(f"Target column '{TARGET_COL}' is still present in X.")

feature_column_table = pd.DataFrame(
    {
        "feature_index": range(len(X.columns)),
        "feature_name": X.columns,
    }
)

print(f"Number of feature columns: {X.shape[1]}")
print("Target column is not present in X. Feature matrix is correctly separated.")
display(feature_column_table)


Number of feature columns: 13
Target column is not present in X. Feature matrix is correctly separated.


,feature_index,feature_name
0,0,V14_V12_interaction
1,1,V14
2,2,V17_V16_interaction
3,3,V12
4,4,V17
5,5,V10
6,6,V4
7,7,V16
8,8,V3
9,9,V11


### Data Type Check

For this fraud dataset, model features should be numeric. This check confirms that every feature column inside `X` is suitable for model training without additional type conversion.


In [23]:
feature_dtype_table = X.dtypes.rename("dtype").reset_index().rename(columns={"index": "feature_name"})
numeric_feature_columns = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_feature_columns = X.select_dtypes(exclude=[np.number]).columns.tolist()

display(feature_dtype_table)
print(f"Numeric feature columns: {len(numeric_feature_columns)}")
print(f"Non-numeric feature columns: {len(non_numeric_feature_columns)}")

if non_numeric_feature_columns:
    print("Non-numeric feature columns detected:")
    print(non_numeric_feature_columns)
    raise TypeError(
        "Non-numeric feature columns found in X. Final model training requires numeric features only."
    )

print("All feature columns are numeric and ready for model training.")


,feature_name,dtype
0,V14_V12_interaction,float64
1,V14,float64
2,V17_V16_interaction,float64
3,V12,float64
4,V17,float64
5,V10,float64
6,V4,float64
7,V16,float64
8,V3,float64
9,V11,float64


Numeric feature columns: 13
Non-numeric feature columns: 0
All feature columns are numeric and ready for model training.


### Dataset Readiness Summary

This summary collects the most important validation checks into one compact table. It is also saved to the reporting folder so the final training workflow has a documented validation checkpoint.


In [24]:
all_features_numeric = len(non_numeric_feature_columns) == 0
target_column_removed_from_X = TARGET_COL not in X.columns
validation_summary_path = TABLES_DIR / "final_training_dataset_validation_summary.csv"

dataset_validation_summary = pd.DataFrame(
    [
        {"check": "dataset_rows", "value": df.shape[0]},
        {"check": "dataset_columns", "value": df.shape[1]},
        {"check": "feature_columns", "value": X.shape[1]},
        {"check": "target_column", "value": TARGET_COL},
        {"check": "total_missing_values", "value": total_missing_values},
        {"check": "duplicate_rows", "value": duplicate_rows},
        {"check": "target_values", "value": str(sorted(actual_target_values))},
        {"check": "normal_transactions", "value": normal_count},
        {"check": "fraud_transactions", "value": fraud_count},
        {"check": "fraud_percentage", "value": round(fraud_percentage, 6)},
        {"check": "imbalance_ratio", "value": round(imbalance_ratio, 6) if np.isfinite(imbalance_ratio) else "inf"},
        {"check": "all_features_numeric", "value": all_features_numeric},
        {"check": "target_column_removed_from_X", "value": target_column_removed_from_X},
    ]
)

dataset_validation_summary.to_csv(validation_summary_path, index=False)
display(dataset_validation_summary)
print(f"Saved dataset validation summary to: {validation_summary_path}")


,check,value
0,dataset_rows,283726
1,dataset_columns,14
2,feature_columns,13
3,target_column,Class
4,total_missing_values,0
5,duplicate_rows,8063
6,target_values,"[0, 1]"
7,normal_transactions,283253
8,fraud_transactions,473
9,fraud_percentage,0.16671


Saved dataset validation summary to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training/final_training_dataset_validation_summary.csv


## Final Training Dataset Validation Notes

The dataset is considered ready for final model training if:

- Missing values are not present
- Target contains only valid labels 0 and 1
- Feature matrix does not contain the target column
- Feature columns are numeric
- Class imbalance is clearly understood and will be considered during model training and evaluation


## Stop and Verify Before Model Training

- Are there any missing values in the final training dataset?
- Are duplicate rows present, and do I understand their count?
- Does the target column contain only 0 and 1?
- How many fraud transactions are present?
- What percentage of the dataset is fraud?
- Is the dataset highly imbalanced?
- Are all feature columns numeric?
- Is the target column removed from X?
- Is the feature column order clear for future inference/API usage?
